# Wykład 3 — Rozmawiamy z programem

## W3.0 — Od wartości do wejścia i wyjścia

Na W1 nauczyliśmy się zapisywać i uruchamiać programy w C; Na W2 poznaliśmy typy, konwersje i operatory. Teraz nauczymy program **przyjmować dane, informować o wyniku i korzystać z gotowych funkcji matematycznych**.

**Cel na dziś:** samodzielnie napisać program, który pobiera liczbę z terminala, sprawdza poprawność danych, wykonuje obliczenie i wyświetla wynik. Po drodze odróżnimy **wydruk** od **wartości zwracanej przez funkcję**.

Plan: `printf` → formatowanie → terminal i strumienie → `scanf` → obsługa błędów → `<math.h>` → zadanie integrujące.

### Przypomnienie z W2: tylko to, czego potrzebujemy

Widzieliśmy już `int`, `double`, `char`, `%d`, `%.2f`, `+`, `/` oraz rzutowanie.

```
int a = 5;
double b = 2.0;
printf("wynik = %.2f\n", a / b);
```

**Przewidź:** co wypisze ten fragment? Dlaczego `a / b` daje wynik zmiennoprzecinkowy? Co zmieniłoby `int b = 2;`?

Dziś zbadamy działanie funkcji `printf`.

### Trzy strumienie, trzy role
Zacznijmy od tego, gdzie program może pisać, oraz skąd pobierać dane:

- `stdin` (*standard input*) — standardowe źródło danych, zwykle klawiatura w terminalu;
- `stdout` (*standard output*) — zwykłe wyniki, zwykle widoczne na ekranie;
- `stderr` (*standard error*) — komunikaty diagnostyczne.

Program nie musi znać urządzenia docelowego: `stdout` można przekierować do pliku. W terminalu przekonamy się o tym za chwilę.

**Ważne:** notebook Jupyter nie jest zwykłym terminalem. Kernel C może obsługiwać standardowe wejście inaczej niż program uruchomiony w konsoli. **Nie uruchamiamy interaktywnego `scanf` w komórce Jupytera**: takie wywołanie może czekać bez końca, dostać EOF lub nie otrzymać danych. Do doświadczeń z parsowaniem zastosujemy `sscanf`, a prawdziwy `scanf` uruchomimy jako osobny program `.c` w terminalu.

## W3.1 — `printf`: coś wypisuje i coś zwraca

Z W1 pamiętamy `#include <stdio.h>`. To tam zadeklarowano funkcję `printf`.

Uproszczony zapis deklaracji: `int printf(const char *format, ...);`. Na tym etapie:

- `format` to tekst sterujący wydrukiem; może zawierać znaczniki `%...`;
- `...` oznacza kolejne argumenty, które zostaną sformatowane;
- zwracany `int` oznacza **liczbę zapisanych znaków przy powodzeniu** lub **wartość ujemną przy błędzie wyjścia**.

`const char *` to zapis zawierający wskaźnik — poznamy go szczegółowo później. **Funkcja, która wypisuje tekst, może równocześnie zwracać wartość**, tak jak funkcja `sum` w W1.

**Przewidź:** ile wynosi wynik pierwszego `printf`, jeśli drukuje tekst `ABC` i znak nowej linii? Czy znak `\n` też się liczy?

In [ ]:
#include <stdio.h>

int main(void)
{
    int zapisano = printf("ABC\n");
    printf("Pierwszy printf zapisal %d znaki.\n", zapisano);
    return 0;
}

**Obserwacja:** pierwszy `printf` zapisał `4` znaki (`A`, `B`, `C` i nową linię). Liczy się to, co trafiło do strumienia, a nie długość zapisu kodu źródłowego (`\n` to jeden znak sterujący). W kodowaniu UTF-8 polska litera może wymagać kilku bajtów i wynik `printf` nie jest wtedy prostą liczbą widocznych liter.

**Eksperyment:** zamień `ABC` na `ABCDE`. Przewidź nową wartość. Wyniku błędu wyjścia nie symulujemy w notebooku, ale wiemy, że jest ujemny.

### Wartość, typ i znacznik formatu muszą do siebie pasować

`printf` nie odczytuje typu argumentu z samej wartości: używa reguł wynikających z formatu.

| Znacznik w `printf` | Oczekiwany argument | Przykładowy wynik |
|:--|:--|:--|
| `%d` lub `%i` | `int` | `-3` |
| `%u`, `%x`, `%o` | `unsigned int` | `12`, `c`, `14` |
| `%f`, `%e`, `%g` | `double` | `2.500000`, `2.500000e+00`, `2.5` |
| `%c` | znak przekazany po promocji jako `int` | `A` |
| `%%` | **bez argumentu** | `%` |

Argument `float` jest w wywołaniu `printf` automatycznie promowany do `double`. Zapis `%lf` dla `printf` jest dozwolony i przy wypisywaniu `double` działa jak `%f`. **Inaczej będzie w `scanf`**, gdzie `%f` i `%lf` dotyczą różnych typów adresów!

**Antyprzykład — nie uruchamiaj:** `printf("%d", 3.14);` nie zmienia typu liczby, lecz narusza zgodność formatu z argumentem i prowadzi do niezdefiniowanego zachowania.

In [ ]:
#include <stdio.h>

int main(void)
{
    int liczba = -3;
    double ulamek = 2.5;
    char litera = 'A';
    unsigned int dodatnia = 12u;

    printf("int=%d, double=%.2f, char=%c\n", liczba, ulamek, litera);
    printf("unsigned=%u, hex=%x, oct=%o\n", dodatnia, dodatnia, dodatnia);
    return 0;
}

**Eksperyment:** zmień `ulamek` z `2.5` na `2.567`, a `%.2f` na `%.1f`. Dlaczego zmienia się **wydruk**, ale nie typ zmiennej? Przypomnij sobie precyzję reprezentacji `double` z W2.

## W3.2 — Formatowanie: szerokość, precyzja i wyrównanie
Wiemy, że wartości liczbowe można formatować za pomocą `%`. Przyjrzyjmy się teraz zapisowi formatowania.

Ogólny schemat: **`%[flagi][szerokość][.precyzja]typ`**.

Najpierw poznajemy trzy części: `%6d`, `%-6d`, `%+d`. Szerokość to **minimalna** szerokość pola, więc duża liczba nie zostanie obcięta. `-` wyrównuje do lewej, `+` wymusza znak liczby ze znakiem.

**Przewidź:** gdzie będą puste miejsca pomiędzy pionowymi kreskami? Policz szerokości samodzielnie.

In [ ]:
#include <stdio.h>

int main(void)
{
    printf("|%6d|%-6d|\n", 12, 12);
    printf("|%6d|%-6d|\n", 1234567, 1234567);
    printf("|%+d|%+d|\n", 12, -12);
    return 0;
}

### Precyzja nie zawsze oznacza to samo

- `%.3d` — wypisz **co najmniej trzy cyfry** liczby całkowitej, z zerami wiodącymi;
- `%.2f` — **dwa miejsca po przecinku** w formacie stałoprzecinkowym;
- `%.2e` — **dwa miejsca po przecinku** w zapisie wykładniczym;
- `%.4g` — do **czterech cyfr znaczących** (zwykle pomija niepotrzebne zera).

Szerokość i precyzja mają inne role. Zapis `%10.2f` oznacza pole szerokie na co najmniej 10 znaków i dwie cyfry po przecinku.

**Przewidź** wynik, potem sprawdź, a na końcu samodzielnie zmień `10` na `6`.

In [ ]:
#include <stdio.h>

int main(void)
{
    double x = 12.3456;
    printf("|%10.2f|\n", x);
    printf("|%.2e|\n", x);
    printf("|%.4g|\n", x);
    printf("|%.3d|\n", 7);
    return 0;
}

### Jeszcze dwa przydatne szczegóły

`%05d` oznacza dopełnianie zerami do szerokości 5. Znak minus wlicza się do szerokości. Aby wypisać dosłowny znak procenta, trzeba napisać `%%` — bez dodatkowego argumentu.

Nie uczymy się dzisiaj na pamięć całej specyfikacji formatów. Najpierw **sprawdź format, a dopiero później kompiluj**; kompilator z `-Wall -Wextra -Werror` często pomoże wychwycić pomyłki.

In [ ]:
#include <stdio.h>

int main(void)
{
    printf("%5.03d\n", 7);
    printf("numer=%05d\n", 7);
    printf("ujemny=%05d\n", -7);
    printf("wypelniono 75%% zadania\n");
    return 0;
}

## W3.3 — Znaki specjalne, terminal i przekierowania

`\n` rozpoczyna nową linię, `\t` to tabulator poziomy, `\\` wypisuje ukośnik odwrotny, `\"` wypisuje cudzysłów w napisie. Nie są to instrukcje **przesuwania kursora**, lecz znaki umieszczane w strumieniu. Ich wizualna interpretacja należy do terminala.

Znak `\r` to *carriage return* (powrót na początek bieżącej linii), a `\b` to *backspace*. Sposób ich wyświetlania może się różnić między terminalem, Jupyterem i plikiem. `\v` to pionowy tabulator, **nie** przenośne „przejście do nowej linii”. Zwykle do kończenia wiersza używamy `\n`.

In [ ]:
#include <stdio.h>

int main(void)
{
    printf("wiersz pierwszy\nwiersz drugi\n");
    printf("A\tB\n");
    printf("Ukosnik: \\ i cudzyslow: \"\n");
    return 0;
}

### Ćwiczenie **w niezależnej konsoli** (nie w komórce Jupytera)

Z katalogu głównego repozytorium:

```
gcc -std=c11 -Wall -Wextra -Werror -pedantic 3/samples/stdio_terminal.c -o w3_stdio
w3_stdio
w3_stdio > w3_stdout.txt
cat w3_stdout.txt
w3_stdio > w3_stdout.txt 2> w3_stderr.txt
cat w3_stderr.txt
```

Zobacz, co trafia do `stdout`, a co do `stderr`. `>` przekierowuje standardowe wyjście, `2>` — standardowy strumień błędów.

**Eksperyment:** co zmieni użycie `fprintf(stderr, "... ");` zamiast `printf`? `fprintf` działa podobnie do `printf`, lecz pozwala wskazać strumień.

## W3.4 — `scanf`: czytanie z `stdin`

`scanf` z `<stdio.h>` próbuje rozpoznać dane według formatu i zapisać je do zmiennych. Uproszczona deklaracja to `int scanf(const char *format, ...);`.

**Najważniejsza różnica:** `printf` **zapisuje tekst**, zaś `scanf` **przypisuje odczytane wartości zmiennym**. Dlatego piszemy `&liczba`: przekazujemy adres zmiennej, do której wynik ma trafić. Operator adresu `&` omówimy szczegółowo przy wskaźnikach.

| Wejście w `scanf` | Oczekiwany adres |
|:--|:--|
| `%d` | `int *`, np. `&a` dla `int a` |
| `%f` | `float *`, np. `&f` dla `float f` |
| `%lf` | `double *`, np. `&x` dla `double x` |
| `%c` | `char *`, np. `&znak` dla `char znak` |

**Wartość zwracana przez `scanf`** to liczba **poprawnie przypisanych pól** (bez pól pomijanych przez `*`). Dla pojedynczego `%d`:

- `1` — udało się przypisać liczbę;
- `0` — napotkano dane, których nie udało się dopasować do liczby;
- `EOF` — wystąpił koniec wejścia lub błąd odczytu **zanim powiodła się pierwsza konwersja**.

`EOF` jest makrem z `<stdio.h>`, zwykle ma wartość `-1`, ale w kodzie porównujemy z `EOF`, nie wpisujemy `-1` na sztywno. **To nie jest liczba przeczytanych znaków**.

### Jak eksperymentować bez ryzyka zawieszenia Jupytera?

Użyjemy `sscanf`, które pobiera dane **z podanego napisu**, nie z klawiatury. Ma te same reguły formatów i analogicznie zwraca liczbę przypisanych pól (lub `EOF`). Samo `sscanf` nie zastępuje interaktywnego `scanf`: pokazuje **parsowanie** niezależnie od terminala.

Przykładowe wywołanie: `sscanf("42", "%d", &a)`.

Nie trzeba jeszcze umieć przechowywać tekstu w tablicach — stały napis to po prostu przygotowane wejście. Do napisów i wskaźników wrócimy później.

**Przewidź:** co zostanie zapisane do `a` i jaka będzie wartość zwrócona?

In [ ]:
#include <stdio.h>

int main(void)
{
    int a = 0;
    int wynik = sscanf("42", "%d", &a);
    printf("sscanf zwrocil %d, a=%d\n", wynik, a);
    return 0;
}

### Dwa pola: sukces to `2`, a nie długość napisu

Format `"%d %lf"` oznacza: najpierw `int`, potem `double`. Spacja w formacie dopuszcza dowolną liczbę białych znaków (także zero). `scanf` może więc rozpoznać `7 2.5`, ale również `7\n2.5`.

**Eksperyment:** zmień zawartość napisu na `"7      2.5"`. Czy zmienia się wynik zwracany? A jeśli wpiszesz `"7"`?

In [ ]:
#include <stdio.h>

int main(void)
{
    int a = 0;
    double b = 0.0;
    int wynik = sscanf("7 2.5", "%d %lf", &a, &b);
    printf("wynik=%d, a=%d, b=%.1f\n", wynik, a, b);
    return 0;
}

### Błędne dane: zwrócone `0`

Jeśli zamiast liczby występują litery, konwersja `%d` nie przypisze niczego. **Nie wolno wtedy używać nieustalonej wartości zmiennej.** Na potrzeby demonstracji wcześniej nadajemy jej wartość `123` i sprawdzamy, czy została zmieniona.

**Przewidź:** czy tekst `kot` można odczytać przez `%d`?

In [ ]:
#include <stdio.h>

int main(void)
{
    int a = 123;
    int wynik = sscanf("kot", "%d", &a);
    printf("wynik=%d, a=%d\n", wynik, a);
    return 0;
}

### Częściowy sukces: zwrócone `1` dla dwóch pól

Jeśli pierwsza konwersja się powiodła, a druga nie, `scanf` i `sscanf` **nie cofają** pierwszego przypisania. Dlatego **wynik `1` nie oznacza pełnego sukcesu**, gdy oczekujemy dwóch pól.

W przykładzie `b` ma wartość początkową `-1.0`, abyśmy mogli bezpiecznie zaobserwować brak przypisania. W zwykłym programie po niepowodzeniu nie należy używać `b` jak poprawnej danej wejściowej.

In [ ]:
#include <stdio.h>

int main(void)
{
    int a = 0;
    double b = -1.0;
    int wynik = sscanf("7 kot", "%d %lf", &a, &b);
    printf("wynik=%d, a=%d, b=%.1f\n", wynik, a, b);
    return 0;
}

### Koniec wejścia: `EOF` zamiast `0`

Jeżeli źródło danych jest puste, nie udało się przeczytać nawet pierwszej wartości: `sscanf` zwraca `EOF`. W terminalu możesz uzyskać analogiczne zachowanie przy pustym przekierowanym pliku lub kombinacji `Ctrl+D` (Linux) na początku wejścia.

**Uwaga:** wartość `EOF` to sygnał o stanie odczytu, nie „liczba znaków”.

In [ ]:
#include <stdio.h>

int main(void)
{
    int a = 123;
    int wynik = sscanf("", "%d", &a);
    printf("wynik=%d, EOF=%d\n", wynik, EOF);
    return 0;
}

### Prawdziwy `scanf` — **obowiązkowo w niezależnej konsoli**

Wykonaj z katalogu głównego repozytorium (tak jak kompilację `hello.c` w W1):

```bash
gcc -std=c11 -Wall -Wextra -Werror -pedantic 3/samples/scanf_stany.c -o /tmp/w3_scanf
/tmp/w3_scanf
```

Podaj `7 2.5`, potem uruchom program ponownie z `7 kot` i `kot`. Na końcu:

```bash
printf '7 2.5\n' | /tmp/w3_scanf
printf '7 kot\n' | /tmp/w3_scanf
printf 'kot\n' | /tmp/w3_scanf
/tmp/w3_scanf < /dev/null
```

Porównaj wypisane `2`, `1`, `0` i `EOF`. Prawidłowe wyniki to **liczby przypisań, nie liczby znaków**. Dla błędnych danych program kończy się statusem niezerowym: to celowe, więc terminal może je oznaczać jako błąd.

## W3.5 — Składnia formatu wejścia i różnice względem `printf`

Najważniejsze pułapki:

- `printf("%f", x)` — oczekuje wartości `double`; `printf("%lf", x)` też jest dopuszczalne;
- `scanf("%f", &f)` — oczekuje **adresu `float`**;
- `scanf("%lf", &x)` — oczekuje **adresu `double`**;
- `%c` **nie pomija** początkowych białych znaków; forma `" %c"` pomija je dzięki spacji w formacie;
- nie dopisuj `\n` na koniec formatu `scanf("%d", &a)`: `scanf("%d\n", &a)` może czekać na kolejny znak niebędący białym znakiem;
- nie stosuj `fflush(stdin)` w przenośnym C; nie dodawaj `%s` bez ograniczenia szerokości przy wczytywaniu do bufora (napisy i buforowanie omówimy później).

**Przewidź:** dlaczego odczyt `double` przez `%f` byłby błędem, chociaż do wypisania `double` wystarcza `%f`?

In [ ]:
#include <stdio.h>

int main(void)
{
    float f = 0.0f;
    double x = 0.0;
    int wynik = sscanf("1.25 2.5", "%f %lf", &f, &x);
    printf("wynik=%d, f=%.2f, x=%.2f\n", wynik, f, x);
    return 0;
}

### `%c` i białe znaki

Pierwszy odczyt pobiera spację z początku napisu. Drugi zapis `" %c"` każe najpierw pominąć białe znaki, a potem odczytać `A`.

**Eksperyment:** usuń spację przed `%c` w drugim wywołaniu. Jaki otrzymasz kod znaku? Przyda się tabela kodów ASCII z W2.8.

In [ ]:
#include <stdio.h>

int main(void)
{
    char pierwszy = '?';
    char drugi = '?';
    int r1 = sscanf(" A", "%c", &pierwszy);
    int r2 = sscanf(" A", " %c", &drugi);
    printf("r1=%d, pierwszy_kod=%d\n", r1, (unsigned char)pierwszy);
    printf("r2=%d, drugi=%c\n", r2, drugi);
    return 0;
}

### Pierwszy krok do bezpiecznej obsługi błędów: `if`

**Nowy element, do którego wrócimy w W4:** `if (warunek) { ... }` wykonuje blok instrukcji tylko wtedy, gdy warunek jest prawdziwy. `!=` z W2.9 oznacza „różne”. Na razie korzystamy z `if` jako prostego zabezpieczenia: **nie używamy wyniku wejścia, gdy parsowanie się nie udało**.

W najbliższym przykładzie celowo podajemy tekst niebędący liczbą. Program nie próbuje wypisać nieodczytanego `a`.

In [ ]:
#include <stdio.h>

int main(void)
{
    int a;
    int wynik = sscanf("kot", "%d", &a);
    if (wynik != 1) {
        printf("Blad danych: oczekiwano int, wynik=%d\n", wynik);
        return 0; /* koniec tego demonstracyjnego programu */
    }
    printf("a=%d\n", a);
    return 0;
}

### Prawdziwy program sprawdza zarówno wejście, jak i warunki zadania

**W niezależnej konsoli** uruchom `3/samples/scanf.c`: program prosi o `int` i `double`. Sprawdź dla `7 2.5`, `7 kot`, `kot` i pustego wejścia. Komunikaty o błędach trafiają do `stderr`, a status zakończenia `return 1;` sygnalizuje niepowodzenie systemowi.

```bash
gcc -std=c11 -Wall -Wextra -Werror -pedantic 3/samples/scanf.c -o /tmp/w3_dane
printf '7 2.5\n' | /tmp/w3_dane
printf '7 kot\n' | /tmp/w3_dane
```

Czy po pierwszej poprawnej liczbie program powinien obliczać wynik, jeśli **druga** wartość jest błędna? Dlaczego nie? Rozbudowaną obsługę wejścia wierszami (np. `fgets` i konwersję) poznamy później — `scanf` nie rozwiązuje wszystkich problemów z obsługą użytkownika.

## W3.6 — `<math.h>`: nie pisz wszystkiego sam

Biblioteka standardowa C zawiera funkcje matematyczne. Deklaracje są dostępne po `#include <math.h>`. `sqrt` oznacza pierwiastek, `pow` potęgowanie, `fabs` wartość bezwzględną dla liczb zmiennoprzecinkowych; `floor` i `ceil` zaokrąglają w dół i w górę do wartości całkowitej **reprezentowanej jako `double`**.

Typowe deklaracje: `double sqrt(double x);`, `double pow(double x, double y);`. Zarówno argumenty, jak i wynik opisano przez `double`.

**Przed uruchomieniem przewidź:** czym różnią się `sqrt(9.0)` oraz `pow(9.0, 0.5)`? Dlaczego w C zapis `2 ^ 3` **nie oznacza potęgowania** (to operator bitowego XOR, który poznamy później)?

Przy kompilacji w terminalu na typowym Linuksie używamy biblioteki matematycznej `-lm` (na końcu polecenia po pliku `.c`). W kernelu C używamy dyrektywy `//%cflags:-lm`, znanej z historycznego notebooka W3. Jeśli konkretny kernel nie obsługuje takiej dyrektywy, uruchom program przez GCC w terminalu.

In [ ]:
//%cflags:-lm
#include <stdio.h>
#include <math.h>

int main(void)
{
    printf("sqrt(9)=%.2f\n", sqrt(9.0));
    printf("pow(2,3)=%.2f\n", pow(2.0, 3.0));
    printf("fabs(-3.5)=%.2f\n", fabs(-3.5));
    printf("floor(2.7)=%.2f, ceil(2.7)=%.2f\n", floor(2.7), ceil(2.7));
    return 0;
}

### Wykładniki i logarytmy: myśl o dziedzinie

- `exp(x)` — oblicza **e do potęgi x**; nie musimy pisać `pow(exp(1.0), x)`;
- `log(x)` — logarytm **naturalny**, czyli przy podstawie `e`;
- `log10(x)` — logarytm dziesiętny.

Dla argumentów rzeczywistych `log(x)` i `log10(x)` wymagają `x > 0`; `sqrt(x)` wymaga `x >= 0`. Biblioteka nie zawsze „bezpiecznie naprawi” niepoprawny argument — może zwrócić `NaN`/nieskończoność i zgłosić błąd matematyczny. Przyjmując dane od użytkownika, **sprawdzamy dziedzinę zanim wywołamy funkcję**.

**Eksperyment:** zastąp `1000.0` przez `100.0` i przewidź wynik `log10`.

In [ ]:
//%cflags:-lm
#include <stdio.h>
#include <math.h>

int main(void)
{
    printf("exp(2)=%.6f\n", exp(2.0));
    printf("log(exp(2))=%.6f\n", log(exp(2.0)));
    printf("log10(1000)=%.6f\n", log10(1000.0));
    return 0;
}

### Trygonometria: **radiany**, nie stopnie

`sin(x)`, `cos(x)` i `tan(x)` przyjmują kąt **w radianach**. `atan2(y, x)` uwzględnia znaki obu współrzędnych i wyznacza kąt do osi x z poprawną ćwiartką, inaczej niż samo `atan(y/x)`.

`M_PI` nie jest gwarantowaną stałą standardowego C11. Liczbę π obliczymy z `acos(-1.0)`. Do zamiany stopni na radiany: `kat_radiany = kat_stopnie * pi / 180.0`.

**Przewidź:** ile wynosi `sin(π/2)`? Co zwróci `atan2(1.0, 1.0)`? Uruchom i porównaj.

In [ ]:
//%cflags:-lm
#include <stdio.h>
#include <math.h>

int main(void)
{
    double pi = acos(-1.0);
    printf("pi=%.6f\n", pi);
    printf("sin(pi/2)=%.6f\n", sin(pi / 2.0));
    printf("atan2(1,1)=%.6f\n", atan2(1.0, 1.0));
    return 0;
}

### Praktyka z `<math.h>` — **w niezależnej konsoli**

Sprawdź gotowe programy z `3/samples`: `math.c` (sinus kąta), `pow.c` (potęga) i `logs.c` (logarytm, tylko dla dodatnich argumentów).

```bash
gcc -std=c11 -Wall -Wextra -Werror -pedantic 3/samples/logs.c -o /tmp/w3_logs -lm
printf '10\n' | /tmp/w3_logs
printf '%s\n' '-5' | /tmp/w3_logs
```

Wywołania z niepoprawnym argumentem powinny zakończyć się **komunikatem diagnostycznym**, nie obliczeniem `log(-5)`.

**Uwaga:** `-lm` w powyższym poleceniu jest **po** pliku źródłowym. W niektórych systemach kolejność bibliotek przy linkowaniu ma znaczenie.

## W3.7 — Łączymy kroki: wejście → kontrola → obliczenie → wyjście

Najpierw wersja powtarzalna w notebooku. Zamiast czekać na klawiaturę, program symuluje wejście `"9"` przez `sscanf`. Używamy prostego `if`, który zabezpiecza przed nieprawidłowymi danymi i przed pierwiastkowaniem wartości ujemnej.

**Przewidź:** co się stanie po zastąpieniu `"9"` przez `"abc"` oraz przez `"-9"`? Zmień jedną rzecz i sprawdź.

In [ ]:
//%cflags:-lm
#include <stdio.h>
#include <math.h>

int main(void)
{
    double liczba;
    int wynik = sscanf("9", "%lf", &liczba);
    if (wynik != 1) {
        printf("Blad: nie odczytano liczby.\n");
        return 1;
    }
    if (liczba < 0.0) {
        printf("Blad: pierwiastek rzeczywisty wymaga liczby nieujemnej.\n");
        return 1;
    }
    printf("sqrt(%.2f)=%.2f\n", liczba, sqrt(liczba));
    return 0;
}

### Wersja z klawiaturą — **wykonaj w niezależnej konsoli**

Teraz prawdziwy program `3/samples/pole_kola.c`:

```bash
gcc -std=c11 -Wall -Wextra -Werror -pedantic 3/samples/pole_kola.c -o /tmp/w3_kolo -lm
/tmp/w3_kolo
printf '2\n' | /tmp/w3_kolo
printf '%s\n' '-2' | /tmp/w3_kolo
printf 'abc\n' | /tmp/w3_kolo
/tmp/w3_kolo < /dev/null
```

- Poprawna dodatnia liczba daje obwód i pole koła.
- Ujemny promień jest odrzucany przed obliczeniem.
- Niepoprawne wejście i EOF nie prowadzą do odczytu niezainicjalizowanej zmiennej.

**Eksperyment:** dodaj do programu średnicę i wypisz ją z precyzją `%.3f`. Jak zmieni się wynik dla promienia `2.5`?

## W3.8 — Samodzielne zadania, pytania kontrolne

**Zadanie A (podstawowe):** sformatuj tabelę trzech wartości `2`, `12`, `125` w polach o szerokości 6. Dopisz nagłówek i dwie cyfry po przecinku dla ich połówek. Najpierw zapisz przewidywany wydruk na kartce.

**Zadanie B (standardowe, terminal):** napisz program, który prosi o **dwie liczby** typu `double`, sprawdza wynik `scanf("%lf %lf", &a, &b)`, a następnie wypisuje ich sumę i średnią. Porównaj zachowanie dla `2 4`, `2 kot`, `kot`, pustego wejścia. Dopiero po sukcesie oblicz średnią.

**Zadanie C (dla chętnych):** rozszerz B o iloraz `a / b`, ale **odrzuć `b == 0.0` przed dzieleniem**. Dodaj `sqrt(a)` tylko dla `a >= 0.0`. Uzasadnij różnicę między odrzuceniem wejścia a ograniczeniem dziedziny funkcji.

**Pytania kontrolne:**

1. Co zwraca `printf` po udanym wydruku, a co w razie błędu? Co zwraca `scanf` przy odczycie `"7 2.5"` dla formatu `"%d %lf"`?
2. Co oznaczają wyniki `0`, `1`, `2` i `EOF` dla **dwóch** pól `scanf`? Dlaczego `1` nie wystarcza?
3. Dlaczego `printf("%f", x)` dla `double` jest poprawne, a `scanf("%f", &x)` już nie?
4. Do czego służy `&` przy `scanf`? Dlaczego nie wolno wypisywać nieodczytanej zmiennej?
5. Dlaczego w Jupyterze użyliśmy `sscanf`, a prawdziwy `scanf` badaliśmy w konsoli?
6. Dlaczego `pow(2.0, 3.0)`, `exp(3.0)` i `2 ^ 3` oznaczają różne rzeczy? Czy `sin(90.0)` oblicza sinus kąta prostego?

## W3.9 — Co dziś zrobiliśmy i co dalej?

| Element | Co potrafimy już wyjaśnić? |
|:--|:--|
| `printf` | Formatowany wydruk na `stdout`; liczba zapisanych znaków lub ujemna wartość przy błędzie. |
| Format | `%d`, `%f`, `%.2f`, szerokość, wyrównanie, `%%`, sekwencje `\n` i `\t`. |
| `scanf` | Przypisanie danych do zmiennych przez ich adresy; liczba przypisanych pól albo `EOF`. |
| Błędne wejście | Rozróżniamy sukces, brak dopasowania, częściowe dopasowanie i brak wejścia; nie używamy nieodczytanych wartości. |
| Jupyter / terminal | `sscanf` dla powtarzalnych doświadczeń, `scanf` dla prawdziwej klawiatury. |
| `<math.h>` | Pierwiastek, potęga, logarytm, funkcje trygonometryczne, radiany i dziedzina; `-lm` przy typowym linkowaniu GCC. |

**Jedna zasada na wynos:** program ma sprawdzić, **czy otrzymał poprawne dane**, zanim zacznie je przetwarzać.

W kolejnym wykładzie wrócimy do `if` jako narzędzia sterowania programem, a potem zaczniemy wykonywać powtarzalne obliczenia za pomocą pętli. To kolejny obrót naszej dydaktycznej spirali.

**Test automatyczny notebooka i przykładów terminalowych:** z katalogu głównego repozytorium uruchom `python3 3/check_w3.py`. Szczegóły w `3/README.md`.